# 02d — Compare Training Results

Aggregates training results from the three per-model notebooks into
cross-model comparison artefacts.

**Prerequisites**: Run all three training notebooks first:
- `02a_train_mobilenet_v3.ipynb` → `reports/history_mobilenet_v3_large.csv`, `reports/timing_mobilenet_v3_large.json`
- `02b_train_resnet50.ipynb` → `reports/history_resnet50.csv`, `reports/timing_resnet50.json`
- `02c_train_swin_tiny.ipynb` → `reports/history_swin_t.csv`, `reports/timing_swin_t.json`

Outputs:
- `reports/curves_comparison.png`
- `reports/training_summary.csv`

## 1 · Environment detection

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

Running in Colab: True


## 2 · Mount Drive & set paths

In [2]:
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah')
else:
    # Local fallback — repo root is parent of scripts/
    DRIVE_ROOT = Path('..').resolve()

(DRIVE_ROOT / 'reports').mkdir(parents=True, exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)

Mounted at /content/drive
DRIVE_ROOT = /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah


## 3 · Import MODEL_NAMES from utils.py

In [3]:
import importlib, shutil, sys

src_utils = DRIVE_ROOT / 'scripts' / 'utils.py'
if src_utils.exists():
    shutil.copy(src_utils, '/content/utils.py' if IN_COLAB else 'utils.py')
    sys.path.insert(0, '/content' if IN_COLAB else '.')
else:
    print(f'[warn] {src_utils} not found — paste utils.py manually.')

import utils  # noqa: E402
importlib.reload(utils)
from utils import MODEL_NAMES

print('MODEL_NAMES:', MODEL_NAMES)

MODEL_NAMES: ('mobilenet_v3_large', 'resnet50', 'swin_t')


## 4 · Load history CSV files

In [4]:
import pandas as pd

NOTEBOOK_MAP = {
    'mobilenet_v3_large': '02a_train_mobilenet_v3.ipynb',
    'resnet50'          : '02b_train_resnet50.ipynb',
    'swin_t'            : '02c_train_swin_tiny.ipynb',
}

histories = {}
for name in MODEL_NAMES:
    csv_path = DRIVE_ROOT / 'reports' / f'history_{name}.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f'Missing: {csv_path}\n'
            f'This file is produced by {NOTEBOOK_MAP[name]}.\n'
            f'Please run that notebook first.'
        )
    histories[name] = pd.read_csv(csv_path)
    print(f'Loaded history_{name}.csv ({len(histories[name])} epochs)')

print(f'\nAll {len(histories)} history files loaded successfully.')

Loaded history_mobilenet_v3_large.csv (13 epochs)
Loaded history_resnet50.csv (14 epochs)
Loaded history_swin_t.csv (14 epochs)

All 3 history files loaded successfully.


## 5 · Load timing JSON files

In [5]:
import json

timings = {}
for name in MODEL_NAMES:
    timing_path = DRIVE_ROOT / 'reports' / f'timing_{name}.json'
    if not timing_path.exists():
        raise FileNotFoundError(
            f'Missing: {timing_path}\n'
            f'This file is produced by {NOTEBOOK_MAP[name]}.\n'
            f'Please run that notebook first.'
        )
    with open(timing_path) as f:
        timings[name] = json.load(f)
    print(f'Loaded timing_{name}.json '
          f'(train_secs={timings[name]["train_secs"]:.1f}, '
          f'best_epoch={timings[name]["best_epoch"]}, '
          f'best_val_acc={timings[name]["best_val_acc"]:.4f})')

print(f'\nAll {len(timings)} timing files loaded successfully.')

Loaded timing_mobilenet_v3_large.json (train_secs=110.3, best_epoch=8, best_val_acc=0.9700)
Loaded timing_resnet50.json (train_secs=102.2, best_epoch=9, best_val_acc=0.9800)
Loaded timing_swin_t.json (train_secs=164.8, best_epoch=9, best_val_acc=0.9800)

All 3 timing files loaded successfully.


## 6 · Plot comparison curves

In [6]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Validation Loss
for name in MODEL_NAMES:
    df = histories[name]
    axes[0].plot(df['epoch'], df['val_loss'], label=name)
axes[0].set(title='Validation Loss per Epoch',
            xlabel='Epoch', ylabel='Validation Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Panel 2: Validation Accuracy
for name in MODEL_NAMES:
    df = histories[name]
    axes[1].plot(df['epoch'], df['val_acc'], label=name)
axes[1].set(title='Validation Accuracy per Epoch',
            xlabel='Epoch', ylabel='Validation Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
comparison_png = DRIVE_ROOT / 'reports' / 'curves_comparison.png'
fig.savefig(comparison_png, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved -> {comparison_png}')

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/curves_comparison.png


## 7 · Compute and write training summary

In [7]:
rows = []
for name in MODEL_NAMES:
    t = timings[name]
    rows.append({
        'model'       : name,
        'best_epoch'  : t['best_epoch'],
        'best_val_acc': t['best_val_acc'],
        'train_minutes': round(t['train_secs'] / 60.0, 2),
    })

summary_df = pd.DataFrame(rows)
summary_csv = DRIVE_ROOT / 'reports' / 'training_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Saved -> {summary_csv}\n')
print(summary_df.to_string(index=False))

Saved -> /content/drive/MyDrive/Penelitian_S1_Hafiz_Amrullah/reports/training_summary.csv

             model  best_epoch  best_val_acc  train_minutes
mobilenet_v3_large           8          0.97           1.84
          resnet50           9          0.98           1.70
            swin_t           9          0.98           2.75


## Done

Cross-model comparison complete. Artefacts saved:
- `reports/curves_comparison.png` — 2-panel figure (val_loss, val_acc)
- `reports/training_summary.csv` — summary table with best_epoch, best_val_acc, train_minutes

Next step: Run `03_evaluate_models.ipynb` for full evaluation metrics.